In [1]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from transformers import (
    SiglipVisionModel, 
    AutoTokenizer, 
    AutoImageProcessor, 
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm
from PIL import Image
from datasets import load_dataset

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
# Define custom linear projection layer
class LinearProjection(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)
        
    def forward(self, x):
        return self.linear(x)

# Define image-text projection model
class ImageTextProjection(nn.Module):
    def __init__(self, image_dim, text_dim):
        super().__init__()
        self.image_projection = nn.Linear(image_dim, text_dim)
        
    def forward(self, x):
        return self.image_projection(x)

# Function to get image embeddings
def get_image_embedding(image, siglip_model, siglip_processor, linear_proj, device):
    with torch.no_grad():
        if isinstance(image, torch.Tensor):
            image = [transforms.ToPILImage()(img.cpu()) for img in image]

        inputs = siglip_processor(images=image, return_tensors="pt")
        inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}
        outputs = siglip_model(**inputs)
        image_features = outputs.pooler_output
        projected_features = linear_proj(image_features)
    return projected_features

In [4]:
# Define image transform
transform = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ToTensor()
])

In [5]:
# Custom Dataset Class for VQA CIFAR-10
class VQACIFAR10Dataset(Dataset):
    def __init__(self, hf_dataset, transform=None, max_samples=None):
        self.dataset = hf_dataset
        self.transform = transform
        self.max_samples = max_samples or len(hf_dataset)

    def __len__(self):
        return min(self.max_samples, len(self.dataset))

    def __getitem__(self, idx):
        data = self.dataset[idx]
        image = data["image"]
        question = str(data["question"])
        answer = str(data["answer"])

        # Apply transform if available
        if self.transform:
            image = self.transform(image)
            
        # Ensure the image is a tensor
        if not isinstance(image, torch.Tensor):
            print(f"Converting image at idx {idx} from {type(image)} to tensor")
            image = transforms.ToTensor()(image)

        return {
            "image": image,
            "question": question,
            "answer": answer
        }

In [6]:
# Custom collate function to combine image, question, and answer
def collate_fn(batch):
    # Check if any items have tensor images
    tensor_images = [item["image"] for item in batch if isinstance(item["image"], torch.Tensor)]
    
    if not tensor_images:
        # If no tensor images, print what we have and convert them
        print(f"Warning: No tensor images found in batch. Converting {len(batch)} items.")
        for i, item in enumerate(batch):
            print(f"Item {i} image type: {type(item['image'])}")
            if not isinstance(item["image"], torch.Tensor):
                # Convert to tensor if it's not already
                if isinstance(item["image"], Image.Image):
                    batch[i]["image"] = transforms.ToTensor()(item["image"])
                else:
                    print(f"Unexpected image type: {type(item['image'])}")
    
    # Now try stacking again
    tensor_images = [item["image"] for item in batch if isinstance(item["image"], torch.Tensor)]
    
    if not tensor_images:
        raise ValueError("No valid tensor images found after conversion attempts")
    
    images = torch.stack(tensor_images)
    questions = [item["question"] for item in batch]
    answers = [item["answer"] for item in batch]
    
    return {
        "image": images,
        "question": questions,
        "answer": answers
    }

In [7]:
# Load SigLIP model and processor
siglip_model = SiglipVisionModel.from_pretrained("google/siglip-so400m-patch14-384").to(device)
siglip_processor = AutoImageProcessor.from_pretrained("google/siglip-so400m-patch14-384")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [8]:
dummy_image = Image.new('RGB', (384, 384), color='black')
with torch.no_grad():
    siglip_inputs = siglip_processor(dummy_image, return_tensors="pt")
    siglip_inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in siglip_inputs.items()}
    siglip_outputs = siglip_model(**siglip_inputs)
    siglip_output_dim = siglip_outputs.pooler_output.shape[-1]

print("Siglip Output Dimensions : ", siglip_output_dim)

Siglip Output Dimensions :  1152


In [9]:
phi_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-128k-instruct",
    device_map="auto"
)
phi_tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-128k-instruct")

if phi_tokenizer.pad_token is None:
    phi_tokenizer.pad_token = phi_tokenizer.eos_token

phi_embed_dim = phi_model.get_input_embeddings().weight.shape[1]

print("Phi Model Embed Dimensions : ", phi_embed_dim)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Phi Model Embed Dimensions :  3072


In [10]:
# Main training function
def train_loop(
    num_images=25000,
    batch_size=128,
    num_epochs=10,
    learning_rate=2e-4,
    dataset_name="TharunSivamani/vqa-cifar10",  # Replace with your HF dataset name
    split="train",
    phi_model=phi_model
):
    
    # Load trained linear projection
    checkpoint = torch.load('linear_projection.pth', map_location=device)
    output_dim = checkpoint['linear.weight'].shape[0]
    print(f"Loading linear projection with output dimension: {output_dim}")
    
    linear_proj = LinearProjection(siglip_output_dim, output_dim).to(device)
    try:
        linear_proj.load_state_dict(checkpoint)
        print("Successfully loaded linear projection weights")
    except Exception as e:
        print(f"Error loading linear projection weights: {e}")
        return
    
    image_text_proj = ImageTextProjection(output_dim, phi_embed_dim).to(device)
    
    phi_model = prepare_model_for_kbit_training(phi_model)

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["o_proj","qkv_proj","down_proj","lm_head","gate_up_proj","qkv_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )

    phi_model = get_peft_model(phi_model, lora_config)
    
    for param in siglip_model.parameters():
        param.requires_grad = False
    for param in linear_proj.parameters():
        param.requires_grad = False

    dataset = load_dataset(dataset_name, split=split)
    dataset = dataset.select(range(num_images))

    # Create a custom dataset instance
    vqa_dataset = VQACIFAR10Dataset(dataset, transform=transform, max_samples=num_images)
    dataloader = DataLoader(vqa_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

    optimizer = AdamW([ 
        {'params': phi_model.parameters()},
        {'params': image_text_proj.parameters()}
    ], lr=learning_rate)

    # Training loop
    for epoch in range(num_epochs):
        total_loss = 0
        phi_model.train()
        image_text_proj.train()
        
        progress_bar = tqdm(dataloader, desc=f'Epoch {epoch+1}/{num_epochs}')
        for batch in progress_bar:
            images = batch["image"].to(device)
            questions_batch = batch["question"]
            answers_batch = batch["answer"]

            image_embeddings = get_image_embedding(images, siglip_model, siglip_processor, linear_proj, device)

            question_tokens = phi_tokenizer(
                questions_batch,
                padding=True,
                truncation=True,
                max_length=256,
                return_tensors="pt"
            ).to(device)

            target_tokens = phi_tokenizer(
                answers_batch,
                padding=True,
                truncation=True,
                max_length=256,
                return_tensors="pt"
            ).to(device)

            question_embeds = phi_model.get_input_embeddings()(question_tokens['input_ids'])
            image_embeds = image_text_proj(image_embeddings).unsqueeze(1)
            combined_embedding = torch.cat([image_embeds, question_embeds], dim=1)

            attention_mask = torch.ones(
                (images.size(0), combined_embedding.size(1)),
                dtype=torch.long,
                device=device
            )

            labels = target_tokens['input_ids'].clone()
            labels = torch.cat([
                torch.full((images.size(0), combined_embedding.size(1) - 1), -100, device=device),
                labels
            ], dim=1)[:, :combined_embedding.size(1)]

            outputs = phi_model(
                inputs_embeds=combined_embedding,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            progress_bar.set_postfix({'loss': loss.item()})

        avg_epoch_loss = total_loss / len(dataloader)
        print(f'Epoch {epoch+1}/{num_epochs}, Average Loss: {avg_epoch_loss:.4f}')

    phi_model.save_pretrained('phi_model_trained')
    torch.save(image_text_proj.state_dict(), 'image_text_proj.pth')
    print("Training completed. Models saved.")

In [11]:
train_loop(
    num_images=50000,
    batch_size=128,
    num_epochs=2,
    dataset_name="TharunSivamani/vqa-cifar10",  # 🔁 Replace this with your dataset
    split="train",
    phi_model=phi_model
)

Loading linear projection with output dimension: 256
Successfully loaded linear projection weights


Epoch 1/2: 100%|██████████| 391/391 [16:43<00:00,  2.57s/it, loss=0.0431]


Epoch 1/2, Average Loss: 0.1470


Epoch 2/2: 100%|██████████| 391/391 [16:22<00:00,  2.51s/it, loss=0.0385]
/home/jovyan/env/lib/python3.12/site-packages/peft/utils/save_and_load.py:220: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


Epoch 2/2, Average Loss: 0.0434
Training completed. Models saved.
